In [ ]:
# ========== 2.1 卷积层理论计算 ==========
import numpy as np

# 输入参数
in_c, in_h, in_w = 3, 32, 32
kernel_num = 16
k_h, k_w = 5, 5
pad = 2
stride = 2

# 1. 计算输出特征图高、宽
out_h = (in_h + 2 * pad - k_h) // stride + 1
out_w = (in_w + 2 * pad - k_w) // stride + 1
out_c = kernel_num

print("===== 卷积层输出尺寸 =====")
print(f"输出通道×高×宽: {out_c} × {out_h} × {out_w}")

# 2. 单个输出像素点乘(乘法)次数
mul_times = in_c * k_h * k_w
print(f"单个输出像素乘法次数: {mul_times}")

# ========== 2.2 手动实现二维最大池化（已修复报错） ==========
def max_pool2d(inputs, kernel_size, stride, padding=0):
    N, C, H, W = inputs.shape
    kh, kw = kernel_size
    
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride

    inputs_pad = np.pad(inputs, ((0,0), (0,0), (padding,padding), (padding,padding)), mode='constant')
    H_pad, W_pad = inputs_pad.shape[2], inputs_pad.shape[3]

    out_h = (H_pad - kh) // sh + 1
    out_w = (W_pad - kw) // sw + 1
    output = np.zeros((N, C, out_h, out_w))

    for n in range(N):
        for c in range(C):
            for h in range(out_h):
                for w in range(out_w):
                    h_start = h * sh
                    h_end = h_start + kh
                    w_start = w * sw
                    w_end = w_start + kw
                    window = inputs_pad[n, c, h_start:h_end, w_start:w_end]
                    output[n, c, h, w] = np.max(window)
    return output

# 测试
if __name__ == "__main__":
    x = np.random.randn(1, 1, 6, 6)
    pool_out = max_pool2d(x, kernel_size=(2,2), stride=2, padding=0)
    print("\n===== 最大池化测试 =====")
    print("输入形状:", x.shape)
    print("池化输出形状:", pool_out.shape)

# ========== 3.1 VGG 卷积参数量计算 ==========
def calc_vgg_params(C):
    params_5x5 = C * 5 * 5 * C
    single_3x3 = C * 3 * 3 * C
    params_two_3x3 = 2 * single_3x3
    return params_5x5, params_two_3x3

C = 64
p5, p3 = calc_vgg_params(C)
print("\n===== VGG 卷积参数量计算 (C={}) =====".format(C))
print(f"单个5×5卷积参数量: {p5}")
print(f"两层串联3×3卷积总参数量: {p3}")

# ========== 3.2 定义 NiN 块 ==========
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                      stride=stride, padding=padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

# 测试
if __name__ == "__main__":
    nin_block = NiNBlock(3, 16, 3, 1, 1)
    x = torch.randn(1, 3, 32, 32)
    y = nin_block(x)
    print("\n===== NiN 块测试 =====")
    print("NiN块输出形状:", y.shape)

# ========== 4.1 批量归一化 BN 计算 ==========
def batch_norm_calc(x_list, gamma, beta, eps):
    x = np.array(x_list)
    mu = np.mean(x)
    var = np.var(x)
    x_hat = (x - mu) / np.sqrt(var + eps)
    y = gamma * x_hat + beta
    return mu, var, x_hat, y

x_data = [2, 4, 6, 8]
gamma = 2
beta = 1
eps = 0

mu, var, x_hat, y_out = batch_norm_calc(x_data, gamma, beta, eps)
print("\n===== 批量归一化计算 =====")
print(f"均值 μ = {mu}")
print(f"方差 σ² = {var}")
print(f"BN输出值 y1~y4: {y_out}")

# ========== 4.2 自定义残差块 Residual ==========
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride) if use_1x1conv else None

    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3 is not None:
            x = self.conv3(x)
        y += x
        return self.relu(y)

# 测试
if __name__ == "__main__":
    res1 = Residual(3, 16, use_1x1conv=True)
    res2 = Residual(16, 16, use_1x1conv=False)
    x = torch.randn(1, 3, 32, 32)
    out1 = res1(x)
    out2 = res2(out1)
    print("\n===== 残差块测试 =====")
    print("残差块1输出形状:", out1.shape)
    print("残差块2输出形状:", out2.shape)

# ========== 5.2 图像增广 Pipeline ==========
from torchvision import transforms

aug_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

print("\n===== 图像增广管道构建完成 =====")
print(aug_pipeline)

# ========== 6.1 IoU 计算 ==========
def calculate_iou(boxA, boxB):
    x1_A, y1_A, x2_A, y2_A = boxA
    x1_B, y1_B, x2_B, y2_B = boxB

    inter_x1 = max(x1_A, x1_B)
    inter_y1 = max(y1_A, y1_B)
    inter_x2 = min(x2_A, x2_B)
    inter_y2 = min(y2_A, y2_B)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_A = (x2_A - x1_A) * (y2_A - y1_A)
    area_B = (x2_B - x1_B) * (y2_B - y1_B)

    union_area = area_A + area_B - inter_area
    iou = inter_area / union_area
    return iou, inter_area, union_area

box_gt = [10, 10, 50, 50]
box_pred = [30, 30, 70, 70]
iou, inter, union = calculate_iou(box_gt, box_pred)
print("\n===== IoU 计算 =====")
print(f"交集面积: {inter}")
print(f"并集面积: {union}")
print(f"IoU 值: {iou:.6f}")

# ========== 6.2 标签平滑交叉熵损失 ==========
import torch.nn.functional as F

def label_smoothing_ce_loss(logits, labels, num_classes, eps=0.1):
    with torch.no_grad():
        smooth_target = torch.full_like(logits, eps / (num_classes - 1))
        smooth_target.scatter_(1, labels.unsqueeze(1), 1 - eps)
    log_probs = F.log_softmax(logits, dim=1)
    loss = -torch.sum(smooth_target * log_probs, dim=1).mean()
    return loss

# 测试
if __name__ == "__main__":
    K = 10
    bs = 4
    logits = torch.randn(bs, K)
    labels = torch.randint(0, K, (bs,))
    loss = label_smoothing_ce_loss(logits, labels, K, 0.1)
    print("\n===== 标签平滑损失 =====")
    print(f"损失值: {loss.item():.6f}")

===== 卷积层输出尺寸 =====
输出通道×高×宽: 16 × 16 × 16
单个输出像素乘法次数: 75

===== 最大池化测试 =====
输入形状: (1, 1, 6, 6)
池化输出形状: (1, 1, 3, 3)

===== VGG 卷积参数量计算 (C=64) =====
单个5×5卷积参数量: 102400
两层串联3×3卷积总参数量: 73728
